

### Cell 1: Setup and Data Preparation


In [ ]:
# Setup and Data Preparation
from google.colab import drive
import os
import shutil
import pandas as pd
import random

drive.mount('/content/drive')

MY_DRIVE_FOLDER = '/content/drive/MyDrive/ML'
TARGET_IMAGES = 1500
LOCAL_DEST_DIR = './dataset'

def smart_setup():
    print(f"Locating data in: {MY_DRIVE_FOLDER}")
    drive_csv = os.path.join(MY_DRIVE_FOLDER, 'Data_Entry_2017.csv')
    drive_img_dir = os.path.join(MY_DRIVE_FOLDER, 'images')

    if not os.path.exists(drive_csv):
        return print(f"Error: CSV not found in {MY_DRIVE_FOLDER}")
    if not os.path.exists(drive_img_dir):
        return print(f"Error: 'images' folder not found in {MY_DRIVE_FOLDER}")

    if os.path.exists(LOCAL_DEST_DIR):
        shutil.rmtree(LOCAL_DEST_DIR)
    os.makedirs(os.path.join(LOCAL_DEST_DIR, 'images'))

    print("Reading CSV and selecting subset...")
    df = pd.read_csv(drive_csv)
    patient_groups = df.groupby('Patient ID')['Image Index'].apply(list).tolist()
    random.shuffle(patient_groups)

    selected_images = []
    current_count = 0

    for p_imgs in patient_groups:
        if current_count >= TARGET_IMAGES:
            break
        valid_batch = [img for img in p_imgs if os.path.exists(os.path.join(drive_img_dir, img))]
        if valid_batch:
            for img in valid_batch:
                shutil.copy2(os.path.join(drive_img_dir, img), os.path.join(LOCAL_DEST_DIR, 'images', img))
            selected_images.extend(valid_batch)
            current_count += len(valid_batch)
            if current_count % 200 == 0:
                print(f"Copied {current_count} images")

    mini_df = df[df['Image Index'].isin(selected_images)]
    mini_df.to_csv(os.path.join(LOCAL_DEST_DIR, 'Data_Entry_2017.csv'), index=False)
    print(f"Data ready at {LOCAL_DEST_DIR}")

smart_setup()

Mounted at /content/drive
Locating data in: /content/drive/MyDrive/ML
Reading CSV and selecting subset...
Copied 200 images
Copied 600 images
Copied 800 images
Data ready at ./dataset


---

### Cell 2: Imports, Configuration, and Dataset Class

we define the core parameters, label mappings, and the PyTorch Dataset class

In [ ]:
import hashlib
import secrets
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.io import read_image
from torchvision.utils import save_image
from diffusers import UNet2DModel, DDPMScheduler

CONFIG = {
    "num_shards": 5,
    "slices_per_shard": 4,
    "image_size": 64,
    "batch_size": 16,
    "train_steps": 50,
    "num_classes": 15
}

LABELS_MAP = {
    'Atelectasis': 0, 'Cardiomegaly': 1, 'Effusion': 2, 'Infiltration': 3,
    'Mass': 4, 'Nodule': 5, 'Pneumonia': 6, 'Pneumothorax': 7,
    'Consolidation': 8, 'Edema': 9, 'Emphysema': 10, 'Fibrosis': 11,
    'Pleural_Thickening': 12, 'Hernia': 13, 'No Finding': 14
}

def get_label_id(label_str):
    return LABELS_MAP.get(label_str.split('|')[0], 14)

class ConditionalXRayDataset(Dataset):
    def __init__(self, metadata_list):
        self.metadata = metadata_list
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
            transforms.Grayscale(),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        path, label_str = self.metadata[idx]
        try:
            return self.transform(read_image(path)), torch.tensor(get_label_id(label_str)).long()
        except:
            return torch.zeros((1, 64, 64)), torch.tensor(14).long()

---

### Cell 3: Model Architectures (Diffusion & Classifier)


In [ ]:
def get_conditional_model():
    return UNet2DModel(
        sample_size=64,
        in_channels=1,
        out_channels=1,
        layers_per_block=1,
        block_out_channels=(32, 64),
        down_block_types=("DownBlock2D", "AttnDownBlock2D"),
        up_block_types=("AttnUpBlock2D", "UpBlock2D"),
        num_class_embeds=CONFIG["num_classes"]
    )

class DiseaseClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        # Corrected in_features from 64 * 32 * 32 to 64 * 16 * 16
        self.classifier = nn.Linear(64 * 16 * 16, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x))

def train_judge(shards, device="cuda" if torch.cuda.is_available() else "cpu"):
    print("\nTraining verifier model")
    model = DiseaseClassifier(CONFIG["num_classes"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    all_data = [item for sublist in shards for item in sublist]
    loader = DataLoader(ConditionalXRayDataset(all_data), batch_size=32, shuffle=True)
    model.train()
    for i in range(15):
        for img, lbl in loader:
            img, lbl = img.to(device), lbl.to(device)
            loss = F.cross_entropy(model(img), lbl)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
    return model

---

### Cell 4: Security and Merkle Ledger


In [ ]:
class SecureMerkleLedger:
    def __init__(self, shards_data):
        self.salts = {}
        self.shards_data = shards_data
        self._build_tree()

    def _hash(self, path):
        if path not in self.salts:
            self.salts[path] = secrets.token_hex(16)
        return hashlib.sha256((os.path.basename(path) + self.salts[path]).encode()).hexdigest()

    def _build_tree(self):
        self.shard_roots = []
        for shard in self.shards_data:
            leaves = [self._hash(item[0]) for item in shard]
            root = hashlib.sha256("".join(sorted(leaves)).encode()).hexdigest() if leaves else "empty"
            self.shard_roots.append(root)
        self.global_root = hashlib.sha256("".join(self.shard_roots).encode()).hexdigest()

    def update_batch(self, deleted_files):
        proofs = []
        for f in deleted_files:
            if f in self.salts:
                proofs.append({"file": os.path.basename(f), "key": self.salts[f]})
                del self.salts[f]
        self._build_tree()
        return self.global_root, proofs

---

### Cell 5: SISA Trainer Class

In [ ]:
class SISATrainer:
    def __init__(self, shards, ledger, verifier):
        self.shards = shards
        self.ledger = ledger
        self.verifier = verifier
        self.checkpoints = {}
        self.model = get_conditional_model().to("cuda" if torch.cuda.is_available() else "cpu")
        self.scheduler = DDPMScheduler(num_train_timesteps=100)
        if os.path.exists("checkpoints"):
            shutil.rmtree("checkpoints")
        os.makedirs("checkpoints")

    def train_shard(self, shard_idx, from_slice=0):
        data = self.shards[shard_idx]
        if not data: return
        slice_size = max(1, len(data) // CONFIG["slices_per_shard"])
        slices = [data[i:i+slice_size] for i in range(0, len(data), slice_size)]

        if shard_idx not in self.checkpoints:
            self.checkpoints[shard_idx] = {}
        device = "cuda" if torch.cuda.is_available() else "cpu"

        for i in range(from_slice, len(slices)):
            loader = DataLoader(ConditionalXRayDataset(slices[i]), batch_size=CONFIG["batch_size"], shuffle=True)
            optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4)
            self.model.train()
            for _ in range(CONFIG["train_steps"]):
                try:
                    img, lbl = next(iter(loader))
                    if img.shape[0] > 0:
                        img, lbl = img.to(device), lbl.to(device)
                        noise = torch.randn_like(img)
                        t = torch.randint(0, 100, (img.shape[0],), device=device)
                        loss = F.mse_loss(
                            self.model(self.scheduler.add_noise(img, noise, t), t, class_labels=lbl).sample,
                            noise
                        )
                        loss.backward()
                        optimizer.step()
                        optimizer.zero_grad()
                except: pass

            path = f"checkpoints/s{shard_idx}_sl{i}.pt"
            torch.save(self.model.state_dict(), path)
            self.checkpoints[shard_idx][i] = path

    def generate_and_verify(self, disease_name):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tid = LABELS_MAP.get(disease_name, 14)
        print(f"\nGenerating {disease_name} sample")
        self.model.eval()
        with torch.no_grad():
            img = torch.randn((1, 1, 64, 64)).to(device)
            lbl = torch.tensor([tid]).long().to(device)
            for t in self.scheduler.timesteps:
                img = self.scheduler.step(
                    self.model(img, t, class_labels=lbl).sample, t, img
                ).prev_sample

            save_image(img, "generated_xray.png", normalize=True)
            print("Saved generated_xray.png")

            self.verifier.eval()
            probs = F.softmax(self.verifier(img), dim=1)
            conf, pred = torch.max(probs, 1)
            pred_name = [k for k,v in LABELS_MAP.items() if v==pred.item()][0]
            print(f"Verifier prediction: {pred_name} ({conf.item()*100:.1f}%)")
            if pred.item() == tid:
                print("Verification passed")
            else:
                print("Verification mismatch")

    def batch_unlearn(self, n=10):
        print(f"\nRemoving {n} samples from training set")
        flat = [(s, item) for s, shard in enumerate(self.shards) for item in shard]
        if len(flat) < n: return
        targets = random.sample(flat, n)

        impact, del_paths = {}, []
        for s_idx, (path, label) in targets:
            try:
                s_data = self.shards[s_idx]
                idx = s_data.index((path, label))
                sl_idx = idx // (max(1, len(s_data) // CONFIG["slices_per_shard"]))
                impact[s_idx] = min(impact.get(s_idx, 999), sl_idx)
                self.shards[s_idx].remove((path, label))
                del_paths.append(path)
            except: continue

        new_root, proofs = self.ledger.update_batch(del_paths)
        print(f"Ledger updated, new root: {new_root[:15]}...")

        with open("audit_log.json", "w") as f:
            json.dump({"type":"BATCH_UNLEARN", "new_root": new_root, "proofs": proofs}, f, indent=2)
        print("Audit log saved to audit_log.json")

        for s_idx, start_slice in impact.items():
            print(f"Retraining shard {s_idx} from slice {start_slice}")
            self.train_shard(s_idx, from_slice=start_slice)

---

### Cell 6: Main Execution Loop


In [ ]:
DATASET_PATH = "./dataset"

if not os.path.exists(os.path.join(DATASET_PATH, "Data_Entry_2017.csv")):
    print("Error: Data not found. Run setup first.")
else:
    df = pd.read_csv(os.path.join(DATASET_PATH, "Data_Entry_2017.csv"))
    all_data = []
    for _, row in df.iterrows():
        p = os.path.join(DATASET_PATH, "images", row['Image Index'])
        if os.path.exists(p):
            all_data.append((p, row['Finding Labels']))
    shards = [all_data[i::CONFIG["num_shards"]] for i in range(CONFIG["num_shards"])]

    if shards:
        judge = train_judge(shards)
        ledger = SecureMerkleLedger(shards)
        print(f"\nGlobal Merkle root: {ledger.global_root[:15]}...")
        trainer = SISATrainer(shards, ledger, judge)

        print("\nInitial training phase")
        for i in range(CONFIG["num_shards"]):
            trainer.train_shard(i)

        print("\nTesting generation")
        trainer.generate_and_verify("Cardiomegaly")

        trainer.batch_unlearn(10)

    print("\nWorkflow complete. Check generated_xray.png and audit_log.json")


Training verifier model

Global Merkle root: 7a098b18c79ff40...

Initial training phase
